# Task 5: Content-Based TED Talks Recommendation Engine

## Objective

The objective of this task is to develop a content-based recommendation system that recommends the five most similar TED Talks based on the textual content of a selected talk.

## Description

In this task, we use the cleaned TED Talks dataset to build a recommendation engine using TF-IDF vectorization and cosine similarity.

We combine the title, tags, description, and transcript of each TED Talk to create text data for recommendation.

TF-IDF converts the text into numerical vectors, and cosine similarity measures how similar two TED Talks are.

A reusable Python function named `recommend_talks(title, top_n=5)` is created to recommend similar talks based on the title entered by the user.

## Steps Performed

1. Loaded the cleaned TED Talks dataset.
2. Combined title, tags, description, and transcript into a single text column.
3. Converted the text into numerical vectors using TF-IDF.
4. Calculated the cosine similarity matrix.
5. Created a reusable `recommend_talks()` function.
6. Excluded the selected TED Talk from its own recommendations.
7. Handled invalid TED Talk titles.
8. Tested the function using different TED Talk titles.
9. Displayed the recommendations in a readable DataFrame.

## Expected Output

The system accepts a TED Talk title and returns the top five similar TED Talks along with their speaker names, URLs, and similarity scores.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from IPython.display import display

In [2]:
df = pd.read_csv(r"C:\Users\vramy\OneDrive\Desktop\samba-shiva-reddy\ted-talks-recommender\data\processed\ted_talks_cleaned.csv")

In [3]:
print("Dataset shape:", df.shape)
print("\nDataset columns:")
print(df.columns.tolist())

display(df.head())

Dataset shape: (2550, 9)

Dataset columns:
['title', 'description', 'tags', 'main_speaker', 'speaker_occupation', 'event', 'views', 'url', 'transcript']


,title,description,tags,main_speaker,speaker_occupation,event,views,url,transcript
0,do schools kill creativity?,sir ken robinson makes an entertaining and pro...,children creativity culture dance education pa...,Ken Robinson,Author/educator,TED2006,47227110,https://www.ted.com/talks/ken_robinson_says_sc...,"good morning. how are you? it's been great, ha..."
1,averting the climate crisis,with the same humor and humanity he exuded in ...,alternative energy cars climate change culture...,Al Gore,Climate advocate,TED2006,3200520,https://www.ted.com/talks/al_gore_on_averting_...,"thank you so much, chris. and it's truly a gre..."
2,simplicity sells,new york times columnist david pogue takes aim...,computers entertainment interface design media...,David Pogue,Technology columnist,TED2006,1636292,https://www.ted.com/talks/david_pogue_says_sim...,"hello voice mail, my old friend. i've called f..."
3,greening the ghetto,"in an emotionally charged talk, macarthur-winn...",macarthur grant activism business cities envir...,Majora Carter,Activist for environmental justice,TED2006,1697550,https://www.ted.com/talks/majora_carter_s_tale...,if you're here today — and i'm very happy that...
4,the best stats you've ever seen,you've never seen data presented like this. wi...,africa asia google demo economics global devel...,Hans Rosling,Global health expert; data visionary,TED2006,12005869,https://www.ted.com/talks/hans_rosling_shows_t...,"about 10 years ago, i took on the task to teac..."


In [4]:
text_columns = ["title", "tags", "description", "transcript"]

for col in text_columns:
    df[col] = df[col].fillna("").astype(str)

df["combined_text"] = (
    df["title"] + " " +
    df["tags"] + " " +
    df["description"] + " " +
    df["transcript"]
).str.strip()

# Keep records that contain some text
talks = df[df["combined_text"].ne("")].copy()

talks.reset_index(drop=True, inplace=True)

print("Total usable TED Talks:", len(talks))

display(talks[["title", "combined_text"]].head())

Total usable TED Talks: 2550


,title,combined_text
0,do schools kill creativity?,do schools kill creativity? children creativit...
1,averting the climate crisis,averting the climate crisis alternative energy...
2,simplicity sells,simplicity sells computers entertainment inter...
3,greening the ghetto,greening the ghetto macarthur grant activism b...
4,the best stats you've ever seen,the best stats you've ever seen africa asia go...


In [5]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=10000,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b"
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    talks["combined_text"]
)

feature_names = tfidf_vectorizer.get_feature_names_out()

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Number of vocabulary terms:", len(feature_names))

print("\nSample vocabulary:")
print(feature_names[:30])

TF-IDF matrix shape: (2550, 10000)
Number of vocabulary terms: 10000

Sample vocabulary:
['aaron' 'ab' 'abandon' 'abandoned' 'abc' 'abed' 'abilities' 'ability'
 'able' 'abnormal' 'abortion' 'abraham' 'abroad' 'absence' 'absent'
 'absolute' 'absolutely' 'absorb' 'absorbed' 'absorbing' 'abstract'
 'abstraction' 'absurd' 'abu' 'abundance' 'abundant' 'abuse' 'abused'
 'abuses' 'abusive']


In [6]:
similarity_matrix = cosine_similarity(tfidf_matrix)

print("Similarity matrix shape:", similarity_matrix.shape)

display(pd.DataFrame(
    similarity_matrix[:5, :5]
))

Similarity matrix shape: (2550, 2550)


,0,1,2,3,4
0,1.000000,0.145005,0.152796,0.099193,0.106056
1,0.145005,1.000000,0.163174,0.140765,0.121826
2,0.152796,0.163174,1.000000,0.107844,0.125087
3,0.099193,0.140765,0.107844,1.000000,0.136107
4,0.106056,0.121826,0.125087,0.136107,1.000000


In [7]:
def recommend_talks(title, top_n=5):

    # Validate input
    if not isinstance(title, str) or not title.strip():
        raise ValueError("Please enter a valid TED Talk title.")

    if not isinstance(top_n, int) or isinstance(top_n, bool) or top_n <= 0:
        raise ValueError("top_n must be a positive integer.")

    # Find the talk by title (case-insensitive)
    matches = talks.index[
        talks["title"].str.strip().str.casefold() == title.strip().casefold()
    ]

    if len(matches) == 0:
        raise ValueError(f"TED Talk title not found: {title}")

    # Get the first matching talk
    talk_index = matches[0]

    # Get similarity scores
    scores = similarity_matrix[talk_index].copy()
    scores[talk_index] = -1

    # Sort and select top N positive scores
    top_indices = np.argsort(scores)[::-1]
    top_indices = [i for i in top_indices if scores[i] > 0][:top_n]

    # Create recommendations
    result = talks.iloc[top_indices][["title", "main_speaker", "url"]].copy()
    result["similarity_score"] = scores[top_indices]

    return result.reset_index(drop=True)


In [8]:
display(
    talks[["title", "main_speaker"]].sample(10, random_state=42)
)

,title,main_speaker
56,"how the ""ghost map"" helped end a killer disease",Steven Johnson
194,juggle and jest,Raspyni Brothers
2225,3 reasons why we can win the fight against pov...,Andrew Youn
233,a hospital tour in nigeria,Seyi Oyesola
1902,the danger of hiding who you are,Morgana Bailey
1211,how to buy happiness,Michael Norton
1894,hopeful lessons from the battle to save rainfo...,Tasso Azevedo
789,the art of the eco-mindshift,Natalie Jeremijenko
1746,the museum of four in the morning,Rives
173,we're worried about local warming ... in your lap,Yossi Vardi


In [9]:
title_1 = talks.iloc[0]["title"]

print("Selected TED Talk:", title_1)

recommendations_1 = recommend_talks(title_1)

display(recommendations_1)

Selected TED Talk: do schools kill creativity?


,title,main_speaker,url,similarity_score
0,bring on the learning revolution!,Ken Robinson,https://www.ted.com/talks/sir_ken_robinson_bri...,0.386260
1,how to escape education's death valley,Ken Robinson,https://www.ted.com/talks/ken_robinson_how_to_...,0.385157
2,how i stopped the taliban from shutting down m...,Sakena Yacoobi,https://www.ted.com/talks/sakena_yacoobi_how_i...,0.307415
3,how to run a company with no rules,Ricardo Semler,https://www.ted.com/talks/ricardo_semler_how_t...,0.306244
4,our failing schools. enough is enough!,Geoffrey Canada,https://www.ted.com/talks/geoffrey_canada_our_...,0.305831


In [10]:
title_2 = talks.iloc[len(talks) // 2]["title"]

print("Selected TED Talk:", title_2)

recommendations_2 = recommend_talks(title_2)

display(recommendations_2)

Selected TED Talk: a tale of mental illness -- from the inside


,title,main_speaker,url,similarity_score
0,there's no shame in taking care of your mental...,Sangu Delle,https://www.ted.com/talks/sangu_delle_there_s_...,0.316554
1,mental health for all by involving all,Vikram Patel,https://www.ted.com/talks/vikram_patel_mental_...,0.271873
2,how electroshock therapy changed me,Sherwin Nuland,https://www.ted.com/talks/sherwin_nuland_on_el...,0.258409
3,what's so funny about mental illness?,Ruby Wax,https://www.ted.com/talks/ruby_wax_what_s_so_f...,0.254457
4,on being just crazy enough,Joshua Walters,https://www.ted.com/talks/joshua_walters_on_be...,0.244703


In [11]:
title_3 = talks.iloc[-1]["title"]

print("Selected TED Talk:", title_3)

recommendations_3 = recommend_talks(title_3)

display(recommendations_3)

Selected TED Talk: how a video game might help us build better cities


,title,main_speaker,url,similarity_score
0,it’s our city. let’s fix it,Alessandra Orofino,https://www.ted.com/talks/alessandra_orofino_i...,0.398095
1,how to protect fast-growing cities from failing,Robert Muggah,https://www.ted.com/talks/robert_muggah_how_to...,0.380534
2,why buses represent democracy in action,Enrique Peñalosa,https://www.ted.com/talks/enrique_penalosa_why...,0.361904
3,brilliant designs to fit more people in every ...,Kent Larson,https://www.ted.com/talks/kent_larson_brillian...,0.357690
4,a song of the city,Jaime Lerner,https://www.ted.com/talks/jaime_lerner_sings_o...,0.348032


In [12]:
try:
    display(recommend_talks("This TED Talk Does Not Exist"))
except ValueError as e:
    print("Error:", e)

Error: TED Talk title not found: This TED Talk Does Not Exist


In [18]:
sample_title = talks.iloc[8]["title"]

print(f"selected title is: {sample_title}")

display(recommend_talks(sample_title.lower()))

selected title is: let's teach religion -- all religion -- in schools


,title,main_speaker,url,similarity_score
0,atheism 2.0,Alain de Botton,https://www.ted.com/talks/alain_de_botton_athe...,0.408535
1,a life of purpose,Rick Warren,https://www.ted.com/talks/rick_warren_on_a_lif...,0.315801
2,why would god create a tsunami?,Tom Honey,https://www.ted.com/talks/tom_honey_on_god_and...,0.274500
3,is religion good or bad?,Kwame Anthony Appiah,https://www.ted.com/talks/kwame_anthony_appiah...,0.255891
4,on technology and faith,Billy Graham,https://www.ted.com/talks/billy_graham_on_tech...,0.241696


In [19]:
sample_title = talks.iloc[40]["title"]

print(f"selected title is: {sample_title}")

display(recommend_talks(sample_title, top_n=3))

selected title is: my wish: manufactured landscapes and green education


,title,main_speaker,url,similarity_score
0,how to run a company with no rules,Ricardo Semler,https://www.ted.com/talks/ricardo_semler_how_t...,0.281148
1,the route to a sustainable future,Alex Steffen,https://www.ted.com/talks/alex_steffen_sees_a_...,0.281046
2,3 warp-speed architecture tales,Bjarke Ingels,https://www.ted.com/talks/bjarke_ingels_3_warp...,0.279611


In [20]:
# Check that the selected talk is not in its own recommendations
selected_title = talks.iloc[10]["title"]
results = recommend_talks(selected_title)

assert selected_title not in results["title"].values
assert len(results) <= 5

print("Self-recommendation check passed.")
print("Number of recommendations:", len(results))

Self-recommendation check passed.
Number of recommendations: 5


# Observations and Limitations

## Observations

1. TF-IDF converts the textual content of TED Talks into numerical vectors.
2. Combining title, tags, description, and transcript provides more information for comparing TED Talks.
3. Cosine similarity helps identify talks with similar textual content.
4. Talks with higher similarity scores are ranked higher in the recommendations.
5. The selected TED Talk is excluded from its own recommendations.
6. The `recommend_talks()` function can recommend talks for different input titles.
7. The function reuses the precomputed similarity matrix, avoiding repeated calculations.
8. The recommendations are displayed in a readable DataFrame.

## Limitations

1. The system depends on the textual content available in the dataset.
2. It may not understand the actual meaning or context of a TED Talk.
3. Talks discussing similar topics with different words may receive lower similarity scores.
4. TF-IDF does not fully understand synonyms or relationships between words.
5. High similarity scores do not always guarantee that the recommendations are useful to every user.

## Conclusion

The content-based recommendation system successfully uses textual information to find similar TED Talks. TF-IDF and cosine similarity help rank talks based on their content. The reusable recommendation function makes it easier to obtain recommendations for different TED Talk titles.